In [13]:
pip install pandas numpy scipy openpyxl xlrd matplotlib lxml html5lib beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\HP'\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import find_peaks, welch
from scipy.interpolate import interp1d
from scipy.integrate import trapezoid

In [15]:
try:
    HERE = Path(__file__).resolve().parent  # if running as .py file
except NameError:
    HERE = Path.cwd()                       # if running in Jupyter

PROJ_ROOT   = HERE
RAW_ROOT    = PROJ_ROOT / "data" / "raw"
OUT_ROOT    = PROJ_ROOT / "data" / "processed"
OUT_DS_ROOT = OUT_ROOT / "downsampled"
OUT_HRV_ROOT= OUT_ROOT / "hrv"

for p in [OUT_DS_ROOT, OUT_HRV_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

In [16]:
GROUP_DIRS = {
    "stress": RAW_ROOT / "Stress induction",
    "conventional": RAW_ROOT / "conventional meditation",
    "vr_high": RAW_ROOT / "high_stress VR meditation",
    "vr_low": RAW_ROOT / "low_stress VR meditation",
}

In [17]:
SKIP_SECONDS = 2      # skip first 2 s to remove irregularities (set to 1 if needed)
MAX_SECONDS  = 240    # keep only first 4 minutes

COLMAP = {
    "Zeit":"time", "U_13_SCL":"SCL", "U_14_SCR":"SCR", "U_15_Temp":"TEMP", "U_15_Tem":"TEMP",
    "U_16_BVP":"BVP", "U_17_PVA":"PVA", "U_18_Puls":"HR", "U_19_Mot":"MOTION",
    "Bemerkung":"NOTE", "Bemerkungen":"NOTE"
}

In [18]:
def read_excel_any(path: Path) -> pd.DataFrame:
    """Robust Excel/HTML reader (handles .xls, .xlsx, .csv)."""
    try:
        return pd.read_excel(path, engine="openpyxl")
    except Exception:
        try:
            return pd.read_excel(path)
        except Exception:
            try:
                tables = pd.read_html(path)
                tables.sort(key=lambda t: t.shape[1], reverse=True)
                return tables[0]
            except Exception:
                raise

In [19]:
def parse_time_to_seconds(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    try:
        hh, mm, rest = s.split(":")
        rest = rest.replace(",", ".")
        return int(hh)*3600 + int(mm)*60 + float(rest)
    except Exception:
        return np.nan

In [20]:
def hrv_from_bvp(t, x):
    """Compute HRV metrics from BVP signal."""
    m = np.isfinite(t) & np.isfinite(x)
    t = np.asarray(t)[m]; x = np.asarray(x)[m]
    if t.size < 10:
        return {"ok": False, "reason": "too few samples"}

    dt = np.median(np.diff(t))
    dt = 0.01 if not np.isfinite(dt) or dt <= 0 else dt
    x_d = x - np.nanmean(x)
    min_dist = max(1, int(0.30 / max(dt, 1e-6)))
    prom = 0.3 * np.nanstd(x_d) if np.isfinite(np.nanstd(x_d)) else None
    peaks, _ = find_peaks(x_d, distance=min_dist, prominence=prom)

    if peaks.size < 3:
        return {"ok": False, "reason": "no peaks"}

    t_peaks = t[peaks]
    rr_all = np.diff(t_peaks)
    ok = (rr_all >= 0.30) & (rr_all <= 2.00)
    rr = rr_all[ok]
    t_rr = t_peaks[1:][ok]

    if rr.size < 3:
        return {"ok": False, "reason": "few valid RR"}

    mean_hr = float(60.0 / np.mean(rr))
    sdnn = float(np.std(rr, ddof=1))
    drr = np.diff(rr)
    rmssd = float(np.sqrt(np.mean(drr**2))) if drr.size > 0 else np.nan
    pnn50 = float(100.0 * np.mean(np.abs(drr) > 0.05)) if drr.size > 0 else np.nan

    lf = hf = lfhf = np.nan
    if t_rr.size >= 4:
        fs = 4.0
        t_uni = np.arange(t_rr.min(), t_rr.max(), 1.0 / fs)
        if t_uni.size >= 16:
            f_i = interp1d(t_rr, rr, kind="linear", bounds_error=False, fill_value="extrapolate")
            rr_i = f_i(t_uni); rr_i = rr_i - np.mean(rr_i)
            f, pxx = welch(rr_i, fs=fs, nperseg=min(256, len(rr_i)))
            LF = (f >= 0.04) & (f < 0.15)
            HF = (f >= 0.15) & (f < 0.40)
            if LF.any():
                lf = float(trapezoid(pxx[LF], x=f[LF]))
            if HF.any():
                hf = float(trapezoid(pxx[HF], x=f[HF]))
            if np.isfinite(lf) and np.isfinite(hf) and hf > 0:
                lfhf = float(lf / hf)
    return {
    "ok": True,
    "n_beats": int(t_peaks.size),
    "mean_HR_bpm": mean_hr,
    "SDNN_s": sdnn,
    "RMSSD_s": rmssd,
    "pNN50_percent": pnn50,
    "LF_power": lf,
    "HF_power": hf,
    "LF_HF_ratio": lfhf,
}


In [21]:
def process_one_file(in_path: Path, out_ds_dir: Path, out_hrv_dir: Path):
    """Process one Excel/CSV file -> save 1s downsample + HRV."""
    out_ds_dir.mkdir(parents=True, exist_ok=True)
    out_hrv_dir.mkdir(parents=True, exist_ok=True)

    df = read_excel_any(in_path)
    df.columns = [str(c).strip() for c in df.columns]

    # Rename columns
    for k, v in COLMAP.items():
        if k in df.columns:
            df.rename(columns={k: v}, inplace=True)

    if "time" not in df.columns:
        raise RuntimeError(f"No Zeit/time column in {in_path.name}")
    # Convert time to seconds
    df["sec"] = df["time"].apply(parse_time_to_seconds)
    df = df[df["sec"].notna()].sort_values("sec").reset_index(drop=True)
    df["sec"] = df["sec"] - df["sec"].min()

    # Skip first 2 s and limit to 4 min
    df = df[(df["sec"] >= SKIP_SECONDS) & (df["sec"] < SKIP_SECONDS + MAX_SECONDS)].copy()
    df["sec"] = df["sec"] - SKIP_SECONDS

    # Numeric conversion
    signals = [c for c in ["SCL","SCR","TEMP","BVP","PVA","HR","MOTION"] if c in df.columns]
    for c in signals:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")

    # 1-second binning
    df["sec_bin"] = np.floor(df["sec"]).astype(int)
    down = df.groupby("sec_bin")[signals].mean().rename_axis("sec").reset_index()
    down = down[down["sec"] < MAX_SECONDS]

    stem = in_path.stem
    down.to_csv(out_ds_dir / f"{stem}.csv", index=False)

    # HRV calculation (BVP preferred)
    if "BVP" in df.columns and df["BVP"].notna().sum() > 0:
        hrv = hrv_from_bvp(df["sec"].to_numpy(), df["BVP"].to_numpy())
    elif "HR" in df.columns and df["HR"].notna().sum() > 0:
        m = np.isfinite(df["HR"].to_numpy()) & (df["HR"].to_numpy() > 0)
        if m.sum() >= 8:
            rr = 60.0 / df["HR"].to_numpy()[m]
            mean_hr = float(60.0 / np.mean(rr))
            sdnn = float(np.std(rr, ddof=1))
            drr = np.diff(rr)
            rmssd = float(np.sqrt(np.mean(drr**2))) if drr.size > 0 else np.nan
            pnn50 = float(100.0 * np.mean(np.abs(drr) > 0.05)) if drr.size > 0 else np.nan
            hrv = {"ok": True, "n_beats": np.nan, "mean_HR_bpm": mean_hr,
                   "SDNN_s": sdnn, "RMSSD_s": rmssd, "pNN50_percent": pnn50,
                   "LF_power": np.nan, "HF_power": np.nan, "LF_HF_ratio": np.nan}
        else:
            hrv = {"ok": False, "reason": "insufficient HR samples"}
    else:
        hrv = {"ok": False, "reason": "no BVP/HR column"}

    pd.DataFrame([hrv]).to_csv(out_hrv_dir / f"{stem}.csv", index=False)



In [22]:
def find_all_data_files(folder: Path):
    """Find .xls, .xlsx, .csv recursively (case-insensitive)."""
    files = []
    for ext in ("*.xls", "*.xlsx", "*.csv", "*.xlsm"):
        files.extend(folder.rglob(ext))
    # Remove duplicates
    uniq = []
    seen = set()
    for f in files:
        fp = str(f.resolve()).lower()
        if fp not in seen:
            seen.add(fp)
            uniq.append(f)
    return sorted(uniq, key=lambda p: str(p).lower())


In [23]:
def find_all_data_files(folder: Path):
    if not folder.exists():
        return []
    files = []
    # recursive patterns
    for pat in ("*.xls", "*.xlsx", "*.xlsm", "*.csv", "*.htm", "*.html", "*.xls.xlsx"):
        files.extend(folder.rglob(pat))
    # de-dupe & normalize
    out = []
    seen = set()
    for f in files:
        key = str(f.resolve()).lower()
        if key not in seen:
            seen.add(key)
            out.append(f)
    return sorted(out, key=lambda p: str(p).lower())

In [24]:
def main():
    for group_name, folder in GROUP_DIRS.items():
        files = find_all_data_files(folder)
        if not files:
            print(f"[{group_name}] No files found under: {folder}")
            continue

        out_ds_dir  = OUT_DS_ROOT / group_name
        out_hrv_dir = OUT_HRV_ROOT / group_name
        out_ds_dir.mkdir(parents=True, exist_ok=True)
        out_hrv_dir.mkdir(parents=True, exist_ok=True)

        print(f"[{group_name}] Found {len(files)} files in {folder}")
        for f in files:
            try:
                process_one_file(f, out_ds_dir, out_hrv_dir)
                print("OK:", group_name, f.name)
            except Exception as e:
                print("FAIL:", group_name, f.name, "->", e)

if __name__ == "__main__":
    main()

[stress] Found 47 files in d:\Git_repository\result folder\data\raw\Stress induction
OK: stress 100.xlsx
OK: stress 101.xlsx
OK: stress 102.xlsx
OK: stress 103.xlsx
OK: stress 104.xlsx
OK: stress 105.xlsx
OK: stress 106.xlsx
OK: stress 107.xlsx
OK: stress 108.xlsx
OK: stress 109.xlsx
OK: stress 110.xlsx
OK: stress 111.xlsx
OK: stress 112.xlsx
OK: stress 113.xlsx
OK: stress 114.xlsx
OK: stress 115.xlsx
OK: stress 116.xlsx
OK: stress 117.xlsx
OK: stress 118.xlsx
OK: stress 119.xlsx
OK: stress 120.xlsx
OK: stress 121.xlsx
OK: stress 122.xlsx
OK: stress 123.xlsx
OK: stress 124.xlsx
OK: stress 125.xlsx
OK: stress 126.xlsx
OK: stress 127.xlsx
OK: stress 128.xlsx
OK: stress 129.xlsx
OK: stress 130.xlsx
OK: stress 131.xlsx
OK: stress 132.xlsx
OK: stress 133.xlsx
OK: stress 134.xlsx
OK: stress 135.xlsx
OK: stress 136.xlsx
OK: stress 137.xlsx
OK: stress 138.xlsx
OK: stress 139.xlsx
OK: stress 140.xlsx
OK: stress 141.xlsx
OK: stress 142.xlsx
OK: stress 143.xlsx
OK: stress 144.xlsx
OK: stress 145.